# RFM Analysis: Customer Value Segmentation

This notebook implements RFM (Recency, Frequency, Monetary) analysis to segment customers based on their purchasing behavior. We'll create RFM scores, tier customers into value segments, and compare RFM segments with the clusters from our previous analysis.

## Table of Contents
1. Data Loading and RFM Feature Engineering
2. RFM Scoring Framework
3. RFM Segmentation and Profiling
4. Customer Value Matrix
5. RFM vs Cluster Analysis
6. Visualizations and Insights
7. Actionable Recommendations

## 1. Data Loading and RFM Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
px.defaults.template = "plotly_white"

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# Load the clustered dataset
df = pd.read_csv("../data/processed/ifood_df_with_clusters.csv")

print(f"Dataset shape: {df.shape}")
print(f"Number of customers: {df.shape[0]:,}")
print(f"\nColumns available: {df.columns.tolist()}")
df.head()

,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,...,education_Basic,education_PhD,MntTotal,MntRegularProds,AcceptedCmpOverall,education_Masters,education_Bachelors,TotalChildren,Cluster,Cluster_Name
0,58138.0,0,0,58,635,88,546,172,88,88,...,0,0,1529,1441,0,0,1,0,2,Empty Nesters
1,46344.0,1,1,38,11,1,6,2,1,6,...,0,0,21,15,0,0,1,2,3,Stretched Parents
2,71613.0,0,0,26,426,49,127,111,21,42,...,0,0,734,692,0,0,1,0,2,Empty Nesters
3,26646.0,1,0,26,11,4,20,10,3,5,...,0,0,48,43,0,0,1,1,1,Young Families on Budget
4,58293.0,1,0,94,173,43,118,46,27,15,...,0,1,407,392,0,0,0,1,1,Young Families on Budget


### RFM Components:

- **Recency (R)**: How recently did the customer make a purchase?
  - Lower recency = Better (more recent purchase)
  - We'll use `Recency` (days since last purchase)

- **Frequency (F)**: How often does the customer make purchases?
  - Higher frequency = Better (more purchases)
  - We'll calculate total purchases across all channels

- **Monetary (M)**: How much does the customer spend?
  - Higher monetary value = Better (more spending)
  - We'll use `MntTotal` (total spending over 2 years)

In [3]:
# Create RFM features
# Recency: Already available as 'Recency' (days since last purchase)
df['R'] = df['Recency']

# Frequency: Total number of purchases across all channels
df['F'] = (df['NumDealsPurchases'] + 
           df['NumWebPurchases'] + 
           df['NumCatalogPurchases'] + 
           df['NumStorePurchases'])

# Monetary: Total spending (already available as MntTotal)
df['M'] = df['MntTotal']

print("RFM Features Created:")
print("="*60)
print(f"\nRecency (R):")
print(df['R'].describe())
print(f"\nFrequency (F):")
print(df['F'].describe())
print(f"\nMonetary (M):")
print(df['M'].describe())

RFM Features Created:

Recency (R):
count    2205.000000
mean       49.009070
std        28.932111
min         0.000000
25%        24.000000
50%        49.000000
75%        74.000000
max        99.000000
Name: R, dtype: float64

Frequency (F):
count    2205.000000
mean       14.887982
std         7.615277
min         0.000000
25%         8.000000
50%        15.000000
75%        21.000000
max        43.000000
Name: F, dtype: float64

Monetary (M):
count    2205.000000
mean      562.764626
std       575.936911
min         4.000000
25%        56.000000
50%       343.000000
75%       964.000000
max      2491.000000
Name: M, dtype: float64


## 2. RFM Scoring Framework

We'll use quintile-based scoring (1-5) for each RFM component:
- **Recency**: 5 = most recent (lowest days), 1 = least recent (highest days)
- **Frequency**: 5 = highest frequency, 1 = lowest frequency
- **Monetary**: 5 = highest spending, 1 = lowest spending

In [4]:
# Create RFM scores using quintiles (1-5 scale)
# For Recency, lower is better, so we reverse the labels
df['R_Score'] = pd.qcut(df['R'], q=5, labels=[5, 4, 3, 2, 1], duplicates='drop')

# For Frequency and Monetary, higher is better
df['F_Score'] = pd.qcut(df['F'], q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')
df['M_Score'] = pd.qcut(df['M'], q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')

# Convert to numeric for calculations
df['R_Score'] = df['R_Score'].astype(int)
df['F_Score'] = df['F_Score'].astype(int)
df['M_Score'] = df['M_Score'].astype(int)

# Calculate overall RFM score (concatenated)
df['RFM_Score'] = df['R_Score'].astype(str) + df['F_Score'].astype(str) + df['M_Score'].astype(str)

# Calculate RFM sum for overall ranking
df['RFM_Sum'] = df['R_Score'] + df['F_Score'] + df['M_Score']

print("RFM Scores Created:")
print("="*60)
print(f"\nScore Distribution:")
print(f"R_Score: {df['R_Score'].value_counts().sort_index().to_dict()}")
print(f"F_Score: {df['F_Score'].value_counts().sort_index().to_dict()}")
print(f"M_Score: {df['M_Score'].value_counts().sort_index().to_dict()}")
print(f"\nRFM_Sum range: {df['RFM_Sum'].min()} to {df['RFM_Sum'].max()}")
print(f"\nSample RFM Scores:")
print(df[['R', 'F', 'M', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'RFM_Sum']].head(10))

RFM Scores Created:

Score Distribution:
R_Score: {1: 434, 2: 448, 3: 431, 4: 441, 5: 451}
F_Score: {1: 543, 2: 406, 3: 385, 4: 460, 5: 411}
M_Score: {1: 443, 2: 441, 3: 439, 4: 441, 5: 441}

RFM_Sum range: 3 to 15

Sample RFM Scores:
    R   F     M  R_Score  F_Score  M_Score RFM_Score  RFM_Sum
0  58  25  1529        3        5        5       355       13
1  38   6    21        4        1        1       411        6
2  26  21   734        4        4        4       444       12
3  26   8    48        4        2        2       422        8
4  94  19   407        1        4        3       143        8
5  16  22   702        5        4        4       544       13
6  34  21   563        4        4        3       443       11
7  32  10   146        4        2        2       422        8
8  19   6    44        5        1        2       512        8
9  68   2    36        2        1        1       211        4


## 3. RFM Segmentation and Profiling

We'll create customer segments based on RFM scores using a tiered approach.

In [5]:
# Define RFM segments based on scores
def rfm_segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    
    # Champions: Best customers (high R, F, M)
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    
    # Loyal Customers: High frequency and monetary, may not be most recent
    elif f >= 4 and m >= 4:
        return 'Loyal Customers'
    
    # Potential Loyalists: Recent customers with average frequency and spending
    elif r >= 4 and f >= 2 and m >= 2:
        return 'Potential Loyalists'
    
    # Recent Customers: Recently active but haven't purchased much yet
    elif r >= 4:
        return 'Recent Customers'
    
    # Promising: Medium recency, frequency, and monetary
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Promising'
    
    # Needs Attention: Above average but slipping
    elif r >= 3 and (f >= 2 or m >= 2):
        return 'Needs Attention'
    
    # At Risk: Used to be good customers but haven't purchased recently
    elif r <= 2 and (f >= 4 or m >= 4):
        return 'At Risk'
    
    # Hibernating: Low recency but had some activity
    elif r <= 2 and f >= 2:
        return 'Hibernating'
    
    # Lost: Lowest scores across the board
    else:
        return 'Lost'

df['RFM_Segment'] = df.apply(rfm_segment, axis=1)

print("RFM Segments Created:")
print("="*60)
segment_counts = df['RFM_Segment'].value_counts()
for segment, count in segment_counts.items():
    pct = count / len(df) * 100
    print(f"{segment}: {count} customers ({pct:.1f}%)")

RFM Segments Created:
Loyal Customers: 436 customers (19.8%)
Potential Loyalists: 378 customers (17.1%)
Lost: 289 customers (13.1%)
Champions: 275 customers (12.5%)
Hibernating: 243 customers (11.0%)
Recent Customers: 239 customers (10.8%)
At Risk: 135 customers (6.1%)
Needs Attention: 106 customers (4.8%)
Promising: 104 customers (4.7%)


In [6]:
# Profile each RFM segment
segment_profile = df.groupby('RFM_Segment').agg({
    'R': 'mean',
    'F': 'mean',
    'M': 'mean',
    'R_Score': 'mean',
    'F_Score': 'mean',
    'M_Score': 'mean',
    'RFM_Sum': 'mean',
    'Income': 'mean',
    'Age': 'mean',
    'AcceptedCmpOverall': 'mean'
}).round(2)

# Add count
segment_profile['Count'] = df.groupby('RFM_Segment').size()
segment_profile['Pct'] = (segment_profile['Count'] / len(df) * 100).round(1)

# Sort by RFM_Sum descending
segment_profile = segment_profile.sort_values('RFM_Sum', ascending=False)

print("\nRFM Segment Profiles:")
print("="*100)
print(segment_profile)


RFM Segment Profiles:
                         R      F        M  R_Score  F_Score  M_Score  \
RFM_Segment                                                             
Champions            19.40  23.59  1151.87     4.49     4.55     4.48   
Loyal Customers      69.52  23.35  1186.94     1.97     4.54     4.54   
Potential Loyalists  18.78  14.00   424.14     4.51     2.78     2.93   
Promising            50.18  17.21   619.47     3.00     3.40     3.45   
At Risk              76.45  17.40   820.88     1.64     3.47     3.72   
Recent Customers     19.28   5.77    34.02     4.52     1.09     1.21   
Needs Attention      49.10   8.98    98.09     3.00     1.73     2.01   
Hibernating          80.19  11.64   174.72     1.47     2.35     2.38   
Lost                 70.87   5.44    34.86     1.91     1.00     1.21   

                     RFM_Sum    Income    Age  AcceptedCmpOverall  Count   Pct  
RFM_Segment                                                                     
Champions  

## 4. Customer Value Matrix

Create a simplified value matrix based on RFM tiers.

In [7]:
# Create simplified value tiers
def value_tier(rfm_sum):
    if rfm_sum >= 12:
        return 'High Value'
    elif rfm_sum >= 9:
        return 'Medium Value'
    elif rfm_sum >= 6:
        return 'Low Value'
    else:
        return 'Very Low Value'

df['Value_Tier'] = df['RFM_Sum'].apply(value_tier)

print("Value Tier Distribution:")
print("="*60)
tier_counts = df['Value_Tier'].value_counts()
tier_order = ['High Value', 'Medium Value', 'Low Value', 'Very Low Value']
for tier in tier_order:
    if tier in tier_counts.index:
        count = tier_counts[tier]
        pct = count / len(df) * 100
        avg_spending = df[df['Value_Tier'] == tier]['M'].mean()
        print(f"{tier}: {count} customers ({pct:.1f}%) - Avg Spending: ${avg_spending:.2f}")

Value Tier Distribution:
High Value: 509 customers (23.1%) - Avg Spending: $1161.04
Medium Value: 718 customers (32.6%) - Avg Spending: $761.45
Low Value: 620 customers (28.1%) - Avg Spending: $141.22
Very Low Value: 358 customers (16.2%) - Avg Spending: $43.73


## 5. RFM vs Cluster Analysis

Let's compare RFM segments with the clusters from Phase 3 to see how they align.

In [8]:
# Create cross-tabulation
crosstab = pd.crosstab(df['Cluster_Name'], df['RFM_Segment'], margins=True)

print("Cross-tabulation: Cluster vs RFM Segment")
print("="*120)
print(crosstab)
print("\n" + "="*120)

Cross-tabulation: Cluster vs RFM Segment
RFM_Segment               At Risk  Champions  Hibernating  Lost  \
Cluster_Name                                                      
Empty Nesters                  55        139            3     0   
Mature Families w/ Teens       53        117           72    30   
Stretched Parents              15         10           69    87   
Young Families on Budget       12          9           99   172   
All                           135        275          243   289   

RFM_Segment               Loyal Customers  Needs Attention  \
Cluster_Name                                                 
Empty Nesters                         232                1   
Mature Families w/ Teens              178               21   
Stretched Parents                      16               33   
Young Families on Budget               10               51   
All                                   436              106   

RFM_Segment               Potential Loyalists  Promisi

In [9]:
# Calculate percentage distribution
crosstab_pct = pd.crosstab(df['Cluster_Name'], df['RFM_Segment'], normalize='index') * 100

print("\nPercentage Distribution: RFM Segment within each Cluster")
print("="*120)
print(crosstab_pct.round(1))
print("\n" + "="*120)


Percentage Distribution: RFM Segment within each Cluster
RFM_Segment               At Risk  Champions  Hibernating  Lost  \
Cluster_Name                                                      
Empty Nesters                10.6       26.8          0.6   0.0   
Mature Families w/ Teens      7.9       17.4         10.7   4.5   
Stretched Parents             3.7        2.5         17.0  21.4   
Young Families on Budget      2.0        1.5         16.3  28.3   

RFM_Segment               Loyal Customers  Needs Attention  \
Cluster_Name                                                 
Empty Nesters                        44.7              0.2   
Mature Families w/ Teens             26.5              3.1   
Stretched Parents                     3.9              8.1   
Young Families on Budget              1.6              8.4   

RFM_Segment               Potential Loyalists  Promising  Recent Customers  
Cluster_Name                                                                
Empty Nester

In [10]:
# Compare clusters by value tier
cluster_value = pd.crosstab(df['Cluster_Name'], df['Value_Tier'], normalize='index') * 100
cluster_value = cluster_value[['High Value', 'Medium Value', 'Low Value', 'Very Low Value']]

print("\nValue Tier Distribution by Cluster:")
print("="*100)
print(cluster_value.round(1))
print("\n" + "="*100)


Value Tier Distribution by Cluster:
Value_Tier                High Value  Medium Value  Low Value  Very Low Value
Cluster_Name                                                                 
Empty Nesters                   52.4          44.5        3.1             0.0
Mature Families w/ Teens        29.4          43.8       20.6             6.3
Stretched Parents                5.9          24.1       42.8            27.3
Young Families on Budget         2.6          15.6       48.0            33.7



## 6. Visualizations and Insights

In [11]:
# RFM Segment distribution
segment_counts = df['RFM_Segment'].value_counts()

fig = px.bar(x=segment_counts.index, y=segment_counts.values,
             title='Customer Distribution by RFM Segment',
             labels={'x': 'RFM Segment', 'y': 'Number of Customers'},
             color=segment_counts.values,
             color_continuous_scale='blues',
             text=segment_counts.values)
fig.update_traces(texttemplate='%{text}<br>(%{y:.1%})', textposition='outside')
fig.update_layout(showlegend=False, height=500, xaxis_tickangle=-45)
fig.show()

In [12]:
# 3D scatter plot of RFM scores
fig = px.scatter_3d(df, x='R_Score', y='F_Score', z='M_Score',
                    color='RFM_Segment',
                    title='RFM 3D Visualization',
                    labels={'R_Score': 'Recency Score',
                           'F_Score': 'Frequency Score',
                           'M_Score': 'Monetary Score'},
                    hover_data=['Income', 'M', 'F', 'R'],
                    opacity=0.7)
fig.update_layout(height=700)
fig.show()

In [13]:
# Value tier distribution
tier_counts = df['Value_Tier'].value_counts()
tier_order = ['High Value', 'Medium Value', 'Low Value', 'Very Low Value']
tier_counts = tier_counts.reindex(tier_order)

colors = ['#2E7D32', '#66BB6A', '#FFA726', '#EF5350']

fig = go.Figure(data=[go.Pie(
    labels=tier_counts.index,
    values=tier_counts.values,
    marker=dict(colors=colors),
    hole=0.4,
    textinfo='label+percent',
    textposition='outside'
)])

fig.update_layout(
    title='Customer Value Tier Distribution',
    height=600,
    annotations=[dict(text='Value<br>Tiers', x=0.5, y=0.5, font_size=20, showarrow=False)]
)
fig.show()

In [14]:
# Heatmap: Average monetary value by Recency and Frequency scores
pivot_data = df.pivot_table(values='M', index='R_Score', columns='F_Score', aggfunc='mean')

fig = go.Figure(data=go.Heatmap(
    z=pivot_data.values,
    x=pivot_data.columns,
    y=pivot_data.index,
    colorscale='YlOrRd',
    text=np.round(pivot_data.values, 0),
    texttemplate='$%{text}',
    textfont={"size": 12},
    colorbar=dict(title="Avg Spending ($)")
))

fig.update_layout(
    title='Customer Value Matrix: Average Spending by Recency & Frequency',
    xaxis_title='Frequency Score (1=Low, 5=High)',
    yaxis_title='Recency Score (1=Old, 5=Recent)',
    height=500,
    width=700
)
fig.show()

In [15]:
# Stacked bar chart: RFM segments within each cluster
crosstab_data = pd.crosstab(df['Cluster_Name'], df['RFM_Segment'])

fig = go.Figure()

for segment in crosstab_data.columns:
    fig.add_trace(go.Bar(
        name=segment,
        x=crosstab_data.index,
        y=crosstab_data[segment],
        text=crosstab_data[segment],
        textposition='inside'
    ))

fig.update_layout(
    title='RFM Segment Distribution Across Customer Clusters',
    xaxis_title='Customer Cluster',
    yaxis_title='Number of Customers',
    barmode='stack',
    height=600,
    xaxis_tickangle=-45,
    legend=dict(title='RFM Segment', orientation="v", yanchor="top", y=1, xanchor="left", x=1.02)
)
fig.show()

In [16]:
# Box plots: RFM component distributions by segment
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=('Recency (days)', 'Frequency (purchases)', 'Monetary ($)'))

segments = df['RFM_Segment'].unique()

for segment in segments:
    segment_data = df[df['RFM_Segment'] == segment]
    
    fig.add_trace(go.Box(y=segment_data['R'], name=segment, showlegend=False), row=1, col=1)
    fig.add_trace(go.Box(y=segment_data['F'], name=segment, showlegend=False), row=1, col=2)
    fig.add_trace(go.Box(y=segment_data['M'], name=segment, showlegend=True), row=1, col=3)

fig.update_layout(height=500, title_text="RFM Component Distributions by Segment")
fig.show()

In [17]:
# Grouped bar chart: Value tier distribution by cluster
tier_by_cluster = df.groupby(['Cluster_Name', 'Value_Tier']).size().unstack(fill_value=0)
tier_order = ['High Value', 'Medium Value', 'Low Value', 'Very Low Value']
tier_by_cluster = tier_by_cluster[tier_order]

fig = go.Figure()

colors_tier = {'High Value': '#2E7D32', 'Medium Value': '#66BB6A', 
               'Low Value': '#FFA726', 'Very Low Value': '#EF5350'}

for tier in tier_order:
    fig.add_trace(go.Bar(
        name=tier,
        x=tier_by_cluster.index,
        y=tier_by_cluster[tier],
        marker_color=colors_tier[tier],
        text=tier_by_cluster[tier],
        textposition='inside'
    ))

fig.update_layout(
    title='Value Tier Distribution by Customer Cluster',
    xaxis_title='Customer Cluster',
    yaxis_title='Number of Customers',
    barmode='group',
    height=600,
    xaxis_tickangle=-45,
    legend=dict(title='Value Tier')
)
fig.show()

## 7. Actionable Recommendations by RFM Segment

In [18]:
# Create comprehensive summary
summary = df.groupby('RFM_Segment').agg({
    'R': 'mean',
    'F': 'mean',
    'M': 'mean',
    'RFM_Sum': 'mean',
    'Income': 'mean',
    'AcceptedCmpOverall': lambda x: (x > 0).sum() / len(x) * 100
}).round(2)

summary['Count'] = df.groupby('RFM_Segment').size()
summary['Pct'] = (summary['Count'] / len(df) * 100).round(1)
summary = summary.sort_values('RFM_Sum', ascending=False)

# Rename columns for clarity
summary.columns = ['Avg Recency', 'Avg Frequency', 'Avg Monetary', 'Avg RFM Sum', 
                   'Avg Income', 'Campaign Response %', 'Count', '%']

print("\n" + "="*120)
print("RFM SEGMENT SUMMARY")
print("="*120)
print(summary)
print("="*120)


RFM SEGMENT SUMMARY
                     Avg Recency  Avg Frequency  Avg Monetary  Avg RFM Sum  \
RFM_Segment                                                                  
Champions                  19.40          23.59       1151.87        13.52   
Loyal Customers            69.52          23.35       1186.94        11.05   
Potential Loyalists        18.78          14.00        424.14        10.22   
Promising                  50.18          17.21        619.47         9.86   
At Risk                    76.45          17.40        820.88         8.83   
Recent Customers           19.28           5.77         34.02         6.82   
Needs Attention            49.10           8.98         98.09         6.74   
Hibernating                80.19          11.64        174.72         6.20   
Lost                       70.87           5.44         34.86         4.13   

                     Avg Income  Campaign Response %  Count     %  
RFM_Segment                                         

### Marketing Strategies by RFM Segment:

#### 🏆 Champions
- **Profile**: Best customers - recent, frequent, high spending
- **Strategy**: 
  - VIP rewards and exclusive experiences
  - Early access to new products
  - Referral incentives
  - Premium customer service

#### 💙 Loyal Customers
- **Profile**: High frequency and spending, may not be most recent
- **Strategy**:
  - Re-engagement campaigns
  - Loyalty program benefits
  - Personalized recommendations
  - "We miss you" offers

#### 🌱 Potential Loyalists
- **Profile**: Recent customers with moderate engagement
- **Strategy**:
  - Onboarding campaigns
  - Product education
  - Frequency incentives
  - Cross-sell opportunities

#### 🆕 Recent Customers
- **Profile**: Recently active but low frequency/spending
- **Strategy**:
  - Welcome series
  - First purchase follow-ups
  - Category exploration campaigns
  - Trial offers

#### 💡 Promising
- **Profile**: Medium scores across all metrics
- **Strategy**:
  - Targeted upsell campaigns
  - Bundle offers
  - Engagement content
  - Feedback requests

#### ⚠️ Needs Attention
- **Profile**: Above average but declining engagement
- **Strategy**:
  - Win-back campaigns
  - Special promotions
  - Satisfaction surveys
  - Limited-time offers

#### 🚨 At Risk
- **Profile**: Previously good customers now inactive
- **Strategy**:
  - Aggressive win-back offers
  - Personal outreach
  - Feedback collection
  - Competitive analysis

#### 😴 Hibernating
- **Profile**: Low recency but some historical activity
- **Strategy**:
  - Reactivation campaigns
  - Deep discounts
  - "Last chance" messaging
  - Product updates

#### 💔 Lost
- **Profile**: Lowest engagement across all metrics
- **Strategy**:
  - Minimal marketing spend
  - Low-cost reactivation attempts
  - Survey for churn reasons
  - Consider removal from active lists

## 8. Export RFM Data

In [19]:
# Export data with RFM scores and segments
output_path = "../data/processed/ifood_df_with_rfm.csv"
df.to_csv(output_path, index=False)

print(f"✓ Data with RFM analysis exported to: {output_path}")
print(f"  Shape: {df.shape}")
print(f"  New columns added: R, F, M, R_Score, F_Score, M_Score, RFM_Score, RFM_Sum, RFM_Segment, Value_Tier")
print(f"\nRFM Segment distribution:")
print(df['RFM_Segment'].value_counts())
print(f"\nValue Tier distribution:")
print(df['Value_Tier'].value_counts())

✓ Data with RFM analysis exported to: ../data/processed/ifood_df_with_rfm.csv
  Shape: (2205, 51)
  New columns added: R, F, M, R_Score, F_Score, M_Score, RFM_Score, RFM_Sum, RFM_Segment, Value_Tier

RFM Segment distribution:
RFM_Segment
Loyal Customers        436
Potential Loyalists    378
Lost                   289
Champions              275
Hibernating            243
Recent Customers       239
At Risk                135
Needs Attention        106
Promising              104
Name: count, dtype: int64

Value Tier distribution:
Value_Tier
Medium Value      718
Low Value         620
High Value        509
Very Low Value    358
Name: count, dtype: int64
